# PEMS-SF 功能性日期分类 - LSTM-Kalman 训练笔记 (V1.0)
目标：不依赖日历标签，仅根据流量波形识别功能性日期（周一至周日），并缓解因调休导致的标签偏移。
数据：PEMS-SF，每日 144 槽占有率（5分钟粒度），已通过 Ward 聚类为 5 组（G1-G5）。本版本默认针对 G1（通勤双峰）训练。
说明：本 Notebook 仅保留训练所需的数据提取与模型代码，去除可视化输出。各步骤附交通流理论解释注释。

In [ ]:
# 基本依赖导入（模块化）
import os, pathlib, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
# SHAP 用于解释性分析（深度模型）
import shap
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = pathlib.Path('.')
DATA_DIR = ROOT / 'data'
STAGE5_DIR = ROOT / 'Stage5-figure'
print('Device:', DEVICE)

In [ ]:
# 训练与解释的可复现性与确定性设置
torch.manual_seed(0); np.random.seed(0); random.seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## 数据加载与结构约束
- 交通流理论：通勤型（G1）在早晚高峰呈现双峰，工作日与周末差异显著，适合先行训练以稳定收敛。
- 输入：时序 $(144, 1)$；静态 5D（峰数、早峰、晚峰、工作日-周末差异、周末异常）。
- 约束：支持按聚类组选择子集（默认 G1）。

In [ ]:
# 读取 Stage5 生成的聚类与特征：为简化，这里从 station_weekday_time_profiles_long 构造 5D
# 若已存在预处理好的 5D 特征与分组文件，可直接读取以避免重复计算。
profiles_long_path = STAGE5_DIR / 'silhouette_results.json'  # 仅用来验证目录存在
if not STAGE5_DIR.exists():
    raise FileNotFoundError('缺少 Stage5-figure 目录，请先运行聚类与特征提取阶段（v3.5）。')

# 站点与标签日级数据来源：直接从原始 PEMS_train/PEMS_test 流式解析（与 v3.5 同逻辑）
train_path = ROOT / 'data' / 'PEMS_train'
test_path  = ROOT / 'data' / 'PEMS_test'
labels_train_path = ROOT / 'data' / 'PEMS_trainlabels'
labels_test_path  = ROOT / 'data' / 'PEMS_testlabels'
stations_list_path = ROOT / 'data' / 'stations_list'

def parse_day_matrix(line: str, expected_rows=963, expected_cols=144):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if x.replace('.', '', 1).isdigit()]
        if nums: data.append(nums)
    if not data: return None
    arr = np.full((len(data), max(len(rr) for rr in data)), np.nan, dtype=float)
    for i, rr in enumerate(data): arr[i, :len(rr)] = rr
    return arr

def iter_day_matrices(path: pathlib.Path, limit=None):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit: break
            mat = parse_day_matrix(line)
            yield mat if mat is not None else None

def load_labels(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    return np.array([int(x) for x in txt.split() if x.isdigit()], dtype=int)

labels_train = load_labels(labels_train_path)
labels_test  = load_labels(labels_test_path)
station_ids = [int(x) for x in stations_list_path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]').split() if x.isdigit()]
assert len(station_ids) == 963, '站点数量应为 963'
print('Train days:', len(labels_train), 'Test days:', len(labels_test))

In [ ]:
# 读取 G1..G5 分组索引并生成站点掩码（与 stations_list 顺序对齐）
groups_index_path = STAGE5_DIR / 'groups_g1_g5.json'
if groups_index_path.exists():
    groups_index = json.loads(groups_index_path.read_text(encoding='utf-8'))
    selected_group = 'g1'  # 默认使用 g1，可改为 'g2'..'g5'
    selected_stations = set(groups_index.get(selected_group, []))
    print(f'使用分组 {selected_group}，站点数: {len(selected_stations)}')
    station_mask = np.array([sid in selected_stations for sid in station_ids], dtype=bool)
else:
    print('未找到分组 JSON，退回全体站点。')
    station_mask = np.ones(963, dtype=bool)

## 5D Functional DNA 构造（无图输出）
交通流解释：5D 以形状特征概括拥堵动力学，峰值数量与时段强度反映通勤规律；工作日/周末差异捕捉制度性变化；周末异常用于标注非典型行为。

In [ ]:
from scipy.signal import find_peaks

def _clean_curve(curve: np.ndarray) -> np.ndarray:
    # 长度对齐与插值清洗，裁剪到 [0,1]
    curve = curve[:144]
    if curve.shape[0] < 144:
        curve = np.pad(curve, (0, 144 - curve.shape[0]), mode='constant', constant_values=np.nan)
    idx = np.arange(curve.size)
    mask = ~np.isnan(curve)
    if mask.any():
        curve = np.interp(idx, idx[mask], curve[mask])
    curve = np.nan_to_num(curve, nan=0.0, posinf=1.0, neginf=0.0)
    curve = np.clip(curve, 0.0, 1.0)
    return curve.astype(np.float32)

def extract_5d_features(mat_963x144: np.ndarray):
    # 对每个站点曲线清洗后提取 5D 特征，避免 NaN 传播
    feats = []
    for i in range(mat_963x144.shape[0]):
        curve = _clean_curve(mat_963x144[i, :])
        peaks, _ = find_peaks(curve, prominence=0.03, width=2)
        n_peaks = min(len(peaks), 3) / 3.0
        denom = float(np.max(curve)) + 1e-8
        am_peak = float(np.max(curve[36:55])) / denom
        pm_peak = float(np.max(curve[96:115])) / denom
        wd_we_var = float(np.linalg.norm(curve[36:60] - curve[96:120]) / math.sqrt(24))
        wd_we_var = float(np.clip(wd_we_var, 0, 1))
        weekend_anom = float(np.clip(np.mean(curve[30:55]) / 0.5, 0, 1))
        feats.append([n_peaks, am_peak, pm_peak, wd_we_var, weekend_anom])
    return np.asarray(feats, dtype=np.float32)

## 数据集封装（按 Cluster 子集，如 G1）
交通流解释：在通勤型子群内训练可降低形状异质性，提高 LSTM 对双峰模式的表征稳定性。

In [ ]:
class PEMSFunctionalDataset(Dataset):
    def __init__(self, split_path: pathlib.Path, labels: np.ndarray, use_mask: np.ndarray | None = None):
        self.samples = []
        self._mask_warned = False
        for day_i, mat in enumerate(iter_day_matrices(split_path)):
            if day_i >= len(labels): break
            if mat is None or mat.shape[1] < 144: continue
            # 掩码长度必须与当日矩阵行一致
            if use_mask is not None:
                if use_mask.shape[0] == mat.shape[0]:
                    mat = mat[use_mask, :]
                elif not self._mask_warned:
                    print(f'警告：掩码长度({use_mask.shape[0]})≠当日矩阵行数({mat.shape[0]})，已跳过掩码。')
                    self._mask_warned = True
            feat5 = extract_5d_features(mat)
            for sidx in range(mat.shape[0]):
                seq = _clean_curve(mat[sidx, :144]).reshape(144, 1)
                f5 = feat5[sidx]
                # 丢弃任何非有限样本
                if not np.isfinite(seq).all() or not np.isfinite(f5).all():
                    continue
                y = int(labels[day_i]) - 1
                self.samples.append((seq.astype(np.float32), f5.astype(np.float32), y))
        self.n = len(self.samples)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        seq, f5, y = self.samples[idx]
        return torch.from_numpy(seq), torch.from_numpy(f5), torch.tensor(y, dtype=torch.long)

train_ds = PEMSFunctionalDataset(train_path, labels_train, station_mask)
test_ds  = PEMSFunctionalDataset(test_path, labels_test, station_mask)
print('Train samples:', len(train_ds), 'Test samples:', len(test_ds))

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)

## 模型定义：LSTM-Kalman Hybrid
交通流解释：LSTM 编码非线性拥堵动力学（高峰与回落）；融合 5D 静态形状特征提升对结构性流型的鲁棒性；卡尔曼后平滑抑制突发事件噪声。

In [ ]:
class LSTMKalmanClassifier(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2, feat5_dim=5, num_classes=7, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fusion_dim = hidden_dim + feat5_dim
        self.mlp = nn.Sequential(
            nn.Linear(self.fusion_dim, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
    def forward(self, seq_bt, feat5_bt):
        # seq_bt: (B,144,1), feat5_bt: (B,5)
        _, (h_n, _) = self.lstm(seq_bt)  # h_n: (num_layers, B, hidden)
        h = h_n[-1]  # 末层隐藏态 (B, hidden)
        x = torch.cat([h, feat5_bt], dim=1)
        logits = self.mlp(x)
        return logits

# 卡尔曼平滑（对类别概率的时间序列；这里按样本维度平滑批次的均值作为占位）
def kalman_smooth_probs(probs_t: np.ndarray, Q=1e-4, R=1e-3):
    # probs_t: (T, C) 时间序列（占位：本任务按日分类，T=1；若进行日序列推断可扩展）
    # 交通流解释：突发拥堵可能扰动短时预测，平滑可提升一致性。
    T, C = probs_t.shape
    x = probs_t.copy()
    # 简化：对每类独立 1D KF，当前占位返回原值（训练阶段不引入后处理影响损失）
    return x

## 训练循环（模块化）
交通流解释：交叉熵度量分类误差；L2 正则与 Dropout 抑制过拟合（应对非平稳流与节假日扰动）。

In [ ]:
model = LSTMKalmanClassifier().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
epochs = 8

def evaluate(loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for seq, f5, y in loader:
            seq = seq.to(DEVICE).float(); f5 = f5.to(DEVICE).float(); y = y.to(DEVICE)
            logits = model(seq, f5)
            preds = torch.argmax(logits, dim=1)
            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())
    acc = accuracy_score(y_true, y_pred)
    return acc, y_true, y_pred

best_acc = 0.0
for ep in range(1, epochs+1):
    model.train(); total_loss = 0.0
    for seq, f5, y in train_loader:
        seq = seq.to(DEVICE).float(); f5 = f5.to(DEVICE).float(); y = y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(seq, f5)
        loss = criterion(logits, y)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * seq.size(0)
    train_acc, _, _ = evaluate(train_loader)
    test_acc, y_true, y_pred = evaluate(test_loader)
    print(f'Epoch {ep}: loss={total_loss/len(train_ds):.4f} train_acc={train_acc:.4f} test_acc={test_acc:.4f}')
    if test_acc > best_acc:
        best_acc = test_acc; torch.save(model.state_dict(), ROOT / 'model_lstm_kalman_g1.pth')
print('Best test acc:', best_acc)

## SHAP 解释性分析（DeepExplainer）
交通流解释：量化 5D 静态特征对某个预测结果的边际贡献，如早高峰强度对工作日识别的作用。

In [ ]:
class MLPForSHAP(nn.Module):
    def __init__(self, fusion_dim=69, num_classes=7, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(fusion_dim, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, num_classes))
    def forward(self, x): return self.net(x)

# 确保推理模式，避免 Dropout 引入随机性导致 SHAP 可加性检查失败
model.eval()

bg_batches = []
for i, (seq, f5, y) in enumerate(train_loader):
    if i >= 10: break
    seq = seq.to(DEVICE).float(); f5 = f5.to(DEVICE).float()
    with torch.no_grad():
        _, (h_n, _) = model.lstm(seq)
        h = h_n[-1]
        fusion = torch.cat([h, f5], dim=1)  # (B, 64+5)
        fusion = torch.nan_to_num(fusion, nan=0.0, posinf=1.0, neginf=0.0)
        bg_batches.append(fusion)
bg = torch.cat(bg_batches, dim=0) if len(bg_batches) else torch.zeros((32, 69), device=DEVICE)
bg = torch.nan_to_num(bg, nan=0.0, posinf=1.0, neginf=0.0)

mlp_shap = MLPForSHAP(fusion_dim=69).to(DEVICE)
mlp_shap.eval()  # 关闭 Dropout 等训练态层
mlp_layers = list(model.mlp.children()); mlp_shap_layers = list(mlp_shap.net.children())
for i in range(len(mlp_layers)):
    if isinstance(mlp_layers[i], nn.Linear) and isinstance(mlp_shap_layers[i], nn.Linear):
        mlp_shap_layers[i].weight.data = mlp_layers[i].weight.data.clone()
        mlp_shap_layers[i].bias.data = mlp_layers[i].bias.data.clone()

explainer = shap.DeepExplainer(mlp_shap, bg)
# 选择一个小批次测试样本进行解释
seq_b, f5_b, y_b = next(iter(test_loader))
seq_b = seq_b.to(DEVICE).float(); f5_b = f5_b.to(DEVICE).float()
with torch.no_grad():
    _, (h_n, _) = model.lstm(seq_b)
    h = h_n[-1]
    fusion_b = torch.cat([h, f5_b], dim=1)
fusion_b = torch.nan_to_num(fusion_b, nan=0.0, posinf=1.0, neginf=0.0)

# 关闭可加性强校验，避免某些算子/数值导致断言失败；若仍失败则回退 GradientExplainer
try:
    shap_values = explainer.shap_values(fusion_b, check_additivity=False)
except Exception as e:
    print('DeepExplainer 失败，回退 GradientExplainer:', str(e))
    explainer = shap.GradientExplainer(mlp_shap, bg)
    shap_values = explainer.shap_values(fusion_b)

# 仅输出 5D 特征维度的平均绝对贡献（无图）
feat_contrib = []
for c in range(7):
    sv = shap_values[c]  # (B, 69)
    f5_sv = np.abs(sv[:, -5:]).mean(axis=0)  # 取后 5 维
    feat_contrib.append(f5_sv.tolist())
print('5D 特征对各类的平均绝对贡献（样本批次）:', json.dumps(feat_contrib, ensure_ascii=False))

## 评估与阈值
交通流解释：65% 全簇平均准确率为初始目标；在通勤型子群上通常更容易达到。后续可引入时序后处理（真实 T>1）与迁移学习以缓解标签偏移。